# Static Stats

Produce basic stats for static GNSS testing, after removing the first 15 minutes (warm up) and last 2 minutes (shut down).

Copyright 2024 Michael George (AKA Logiqx).

This file is part of [GPS Wizard](https://github.com/Logiqx/gps-wizard) and is distributed under the terms of the GNU General Public License.

GPS Wizard is free software: you can redistribute it and/or modify it under the terms of the GNU General Public License as published by the Free Software Foundation, either version 3 of the License, or (at your option) any later version.

GPS Wizard is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY; without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the GNU General Public License for more details.

You should have received a copy of the GNU General Public License along with GPS Wizard. If not, see <https://www.gnu.org/licenses/>.

## Notes

This is very crude and simply looks for Doppler speeds in excess of 30 m/s (i.e. 108 km/h or 58.32 knots).

Such speeds are currently unrealistic for a windsurfer.

In [10]:
import os
import sys

import numpy as np

import traceback

corePath = os.path.join('..', 'core')
if corePath not in sys.path:
    sys.path.extend([corePath])

from file_reader import getFileReader

## Main Function

In [11]:
WARM_UP_SECS = 1800
SHUTDOWN_SECS = 300

MS_TO_KNOTS = 3600 / 1852

def getInterval(data):
    '''Determine the typical interval (milliseconds) between timestamps'''

    tsDiffs = ((data['ts'][1:] - data['ts'][:-1]) * 1000).round().astype(int)
    uniqueValues, counts = np.unique(tsDiffs, return_counts=True)
    interval = uniqueValues[np.argmax(counts)]
    
    return interval


def main():
    '''Iterate through sessions testing each GPS file'''

    rootDir = os.path.join(projdir, '..', 'ubx-cfg', 'gpslogs', 'syrac', 'test-4')

    errors = {}
    spikes = {}
    
    totFiles = 0
    totPoints = 0
    totDist = 0

    print('dir,file,points,rate,min,max,mean,med,std')
    
    for root, subDirs, files in os.walk(rootDir):
        for file in sorted(files):
            ext = os.path.splitext(file)[1].lower()

            # Only interested in some file types
            if ext and ext in ['.sbp', '.oao']:
                filePath = os.path.join(root, file)
                reader = getFileReader(filePath)
                try:
                    reader.load()
                    
                    # Process all tracks within the file
                    for track in reader.tracks:
                        interval = getInterval(track.data)
                        rate = round(1000 / interval)

                        minTs = np.min(track.data['ts'])
                        maxTs = np.max(track.data['ts'])
                        
                        points = (np.argwhere((track.data['ts'] >= minTs + WARM_UP_SECS) & (track.data['ts'] <= maxTs - SHUTDOWN_SECS)))

                        # e.g. sog, sat, or ehve
                        metric = 'ehve'

                        minVal = np.min(track.data[metric][points]) * MS_TO_KNOTS
                        maxVal = np.max(track.data[metric][points]) * MS_TO_KNOTS
                        meanVal = np.mean(track.data[metric][points]) * MS_TO_KNOTS
                        medVal = np.median(track.data[metric][points]) * MS_TO_KNOTS
                        stdVal = np.std(track.data[metric][points]) * MS_TO_KNOTS

                        print('{},{},{},{},{:.3f},{:.3f},{:.5f},{:.5f},{:.5f}'.format(
                            os.path.basename(root), file, len(points), rate, minVal, maxVal, meanVal, medVal, stdVal))

                    totFiles += 1

                except Exception:
                    errors[filePath.replace(projdir + '/', '')] = traceback.format_exc()
                    print('E', end='')

    print(os.linesep * 2 + 'Summary: files = {}'.format(totFiles))

    if len(errors) > 0:
        print(os.linesep * 2 + 'Errors:')
        for error in errors:
            print(error)
            print(errors[error])

In [12]:
if __name__ == '__main__':
    projdir = os.path.realpath(os.path.join(sys.path[0], "..", ".."))

    main()
    
    print(os.linesep + 'All done!')

dir,file,points,rate,min,max,mean,med,std
test-7,D1_2607140032.sbp,317281,10,0.058,0.156,0.11585,0.11663,0.01391
test-7,D2__2607140033.sbp,331097,10,0.058,0.175,0.12725,0.13607,0.01535
test-7,D3__2607140033.sbp,165637,5,0.097,0.233,0.16631,0.17495,0.02248
test-7,D5_2607140031.sbp,165986,5,0.078,0.253,0.17360,0.17495,0.02256
test-7,SY1__2607140034.sbp,330209,10,0.058,0.194,0.12980,0.13607,0.01416
test-7,SY2_2607140033.sbp,165572,5,0.078,0.253,0.14792,0.15551,0.02125
test-7,ubxGPS_2607140032.sbp,165617,5,0.078,0.350,0.16596,0.17495,0.02261


Summary: files = 7

All done!
